# CIFAR-10 Long-Tail (LT) 불균형 분류 실험

---

## 1. 태스크 및 도메인
- **도메인**: CIFAR-10 Long-Tail (인공 불균형) 이미지 분류
- **모달리티**: RGB 컬러 이미지 (32×32)
- **태스크**: 10-class 분류
  - 클래스: 비행기, 자동차, 새, 고양이, 사슴, 개, 개구리, 말, 배, 트럭
- **핵심 도전**: 극단적 클래스 불균형 (IR=10/50/100) 하에서 손실 함수 효과 검증

## 2. 모델
- **아키텍처**: ResNet-32 (He et al. 2016 CIFAR 전용)
- **사전학습**: 없음 (CIFAR-LT는 scratch 학습이 표준)
- **선택 이유**: CIFAR-LT 벤치마크 표준 모델 (LDAM, BBN, cRT 등 논문 기준선)
- **출력**: 10채널 softmax logits

## 3. 데이터셋
- **이름**: CIFAR-10 Long-Tail (torchvision + 지수 감소 서브샘플링)
- **규모**: 
  - Original CIFAR-10: 50,000 train / 10,000 test (클래스당 5,000 / 1,000)
  - LT 변환: IR=100 → train 클래스당 5,000~50장 (불균형), test는 균형 유지
- **입력 해상도**: 32×32 RGB
- **클래스 불균형**: 
  - IR=10: max 5,000 / min 500 (비교적 완만)
  - IR=50: max 5,000 / min 100
  - IR=100: max 5,000 / min 50 (극심)
- **공식 분할**: 없음 → 8:1:1 (train 40,000 / val 5,000 / test 10,000)

## 4. 데이터 준비 (협업자용)
> Cell 0 자동 실행 시 torchvision으로 자동 다운로드됩니다.

**취득 방법**:
- `torchvision.datasets.CIFAR10(download=True)` — Cell 0 실행 시 자동 다운로드

**Colab 환경**: 설치 불필요 (torchvision 사전 설치됨)

## 5. 전처리 및 데이터 특이점
- **Augmentation (학습)**: RandomCrop(32, padding=4) + RandomHorizontalFlip
- **정규화**: ImageNet 기준 mean/std 사용 (CIFAR-LT 논문 표준)
  - mean=[0.4914, 0.4822, 0.4465]
  - std=[0.2023, 0.1994, 0.2010]
- **불균형 생성**: 지수 감소 분포 — n_i = n_max × IR^(-i/(K-1))
  - n_max = 5,000 (원본 클래스당 샘플 수)
  - K = 10 (클래스 수)
- **Test Set**: 원본 균형 유지 (각 클래스 1,000장) — 공정한 평가

## 6. 실험 손실 함수 및 Optuna 탐색 범위
| 손실 함수 | 탐색 파라미터 | 탐색 범위 | Trials |
|-----------|-------------|----------|--------|
| `ce` | — | — | — |
| `wce` | — | — | — |
| `lwce` | — | — | — |
| `plwce` | alpha | 2.5 ~ 15.0 | 20 (1D GridSampler) |
| `cb` | — | — | — |
| `plwce_focal` | alpha + gamma | alpha 2.5~15.0(8) × gamma 0.5~5.0(5) | 40 (2D GridSampler) |

**Optuna 설정** (proxy learning):
- subset_ratio=0.20 (전체 LT 학습셋의 20%만 사용)
- proxy_epochs=40 (빠른 탐색)
- metric: Balanced Accuracy (macro recall, 불균형 환경에서 핵심)

## 7. SoTA 참고 (2025년 12월 기준)
| 방법 | Top-1 Acc (%) | Balanced Acc (%) | Few-shot Acc (%) | 출처 |
|------|-------------|-----------------|------------------|------|
| LDAM (2019) | 72.58 (IR=100) | — | — | ICML'19 |
| BBN (2019) | 73.41 (IR=100) | — | — | ICCV'19 |
| cRT (2020) | 75.19 (IR=100) | — | — | ICML'21 |
| Decoupling (2019) | 76.46 (IR=100) | — | — | ICCV'19 |
| 임의 U-Net (CIFAR-LT 비표준) | ~65 (IR=100) | — | — | baseline |

> 본 연구 목표: ResNet-32 + 표준 SGD 학습 하에서 LWCE/PLWCE 손실함수의 개선 효과 검증.
> 평가 지표: Top-1 정확도 + Balanced Accuracy (macro recall) + Few-shot Accuracy
> 결과 저장: `image_classification/results/CIFAR10_LT/IR{ir}/`


In [1]:
# === Cell 0: 환경 설정 ===

!pip install optuna torchvision pandas openpyxl -q

import os, sys, json, pickle
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
import torchvision.transforms as transforms
import torchvision.datasets as datasets
from sklearn.metrics import confusion_matrix, f1_score

import optuna
from optuna.samplers import GridSampler

# --- Google Drive 마운트 (Colab only) ---
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
    print('✓ Google Drive 마운트 완료')
except:
    IN_COLAB = False
    print('✓ 로컬/하이브리드 환경에서 실행 중')

# --- 모듈 경로 설정: custom_losses.py와 resnet32.py 찾기 ---
IMG_CLF_DIR = os.getcwd()
if not os.path.exists(f'{IMG_CLF_DIR}/custom_losses.py'):
    if IN_COLAB:
        IMG_CLF_DIR = '/content/drive/MyDrive/imbalanced-data-LWCE/image_classification'
    else:
        IMG_CLF_DIR = os.path.dirname(os.path.abspath(__file__)) if '__file__' in dir() \
                      else 'C:/Users/Seung/Desktop/Research/Deep_Learning/imbalanced-data-LWCE/image_classification'

if IMG_CLF_DIR not in sys.path:
    sys.path.insert(0, IMG_CLF_DIR)

try:
    from custom_losses import get_clf_loss
    from resnet32 import build_resnet32
    print(f'✓ 모듈 로드 성공: {IMG_CLF_DIR}')
except ModuleNotFoundError as e:
    print(f'⚠️  모듈 로드 실패: {e}')
    raise

# --- 상수 설정 ---
DATASET = 'cifar10'
NUM_CLASSES = 10
IR_LIST = [10, 50, 100]

BATCH_SIZE = 128
NUM_WORKERS = 0
SEED = 42
FINAL_EPOCHS = 200
SEEDS = [42, 43, 44, 45, 46]

# 결과 저장 경로
if IN_COLAB:
    RESULTS_BASE = '/content/drive/MyDrive/imbalanced-data-LWCE/image_classification/results/CIFAR10_LT'
else:
    RESULTS_BASE = './results/CIFAR10_LT'

os.makedirs(RESULTS_BASE, exist_ok=True)

CKPT_OPTUNA  = f'{RESULTS_BASE}/optuna_checkpoint.json'
CKPT_RESULTS = f'{RESULTS_BASE}/results_checkpoint.json'

# --- 디바이스 설정 ---
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'✓ Device: {device}')
if torch.cuda.is_available():
    print(f'  GPU: {torch.cuda.get_device_name(0)}')
    print(f'  Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

# --- 초기 시드 고정 ---
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)

# --- Matplotlib 백엔드 ---
import matplotlib
matplotlib.use('Agg')

print('\n✓ 환경 설정 완료')
print(f'  Seeds: {SEEDS}')
print(f'  Total runs: {len(IR_LIST)} IRs × 6 losses × {len(SEEDS)} seeds = {len(IR_LIST)*6*len(SEEDS)} runs')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✓ Google Drive 마운트 완료
✓ 모듈 로드 성공: /content/drive/MyDrive/imbalanced-data-LWCE/image_classification
✓ Device: cuda
  GPU: NVIDIA L4
  Memory: 23.7 GB

✓ 환경 설정 완료
  Seeds: [42, 43, 44, 45, 46]
  Total runs: 3 IRs × 6 losses × 5 seeds = 90 runs


In [ ]:
# === Cell 1: CIFAR-LT 데이터셋 생성 및 로드 ===

def make_cifar_lt(dataset_name: str, imbalance_ratio: int, seed: int = 42):
    """
    CIFAR 데이터셋을 불균형(long-tail) 분포로 변환.
    
    지수 감소: n_i = n_max × IR^(-i/(K-1))
    
    Args:
        dataset_name: 'cifar10' or 'cifar100'
        imbalance_ratio: IR=10, 50, 100 등
        seed: 재현성
        
    Returns:
        indices (train LT indices), class_counts (list of K elements)
    """
    K = 10 if dataset_name == 'cifar10' else 100
    n_max = 5000 if dataset_name == 'cifar10' else 500
    
    # 전체 데이터셋 다운로드
    if dataset_name == 'cifar10':
        dataset = datasets.CIFAR10(root='/tmp/cifar', train=True, download=True, transform=None)
    else:
        dataset = datasets.CIFAR100(root='/tmp/cifar', train=True, download=True, transform=None)
    
    targets = np.array(dataset.targets)
    
    # 클래스별 인덱스 그룹화
    class_indices = [np.where(targets == c)[0] for c in range(K)]
    
    # 불균형 수정: n_i 계산
    rho = imbalance_ratio ** (-1 / (K - 1))
    class_counts = [int(n_max * (rho ** i)) for i in range(K)]
    
    # 각 클래스에서 n_i개씩 랜덤 선택
    np.random.seed(seed)
    lt_indices = []
    for c, n_samples in enumerate(class_counts):
        n_samples = max(1, n_samples)  # 최소 1개
        selected = np.random.choice(class_indices[c], size=n_samples, replace=False)
        lt_indices.extend(selected)
    
    lt_indices = np.array(lt_indices)
    np.random.shuffle(lt_indices)
    
    return lt_indices.tolist(), class_counts


# --- CIFAR-10 train/val/test splits ---
def load_cifar_lt_loaders(ir: int, batch_size: int = 128, num_workers: int = 0):
    """
    CIFAR-10 LT + standard CIFAR-10 test을 로드.
    Train LT set을 80/20으로 나눔 (val은 Optuna/early stopping용, test는 최종 평가용)
    """
    # LT train 생성
    full_dataset = datasets.CIFAR10(root='/tmp/cifar', train=True, download=True, transform=None)
    lt_indices, class_counts = make_cifar_lt('cifar10', ir, seed=SEED)
    lt_indices = np.array(lt_indices)  # Convert to numpy array for proper indexing
    
    # train/val 분할 (stratified)
    lt_targets = np.array(full_dataset.targets)[lt_indices]
    n_val = len(lt_indices) // 5  # 20%
    
    # 클래스별로 stratified split
    train_indices, val_indices = [], []
    for c in range(10):
        c_mask = lt_targets == c
        c_idx = np.where(c_mask)[0]
        np.random.seed(SEED)
        np.random.shuffle(c_idx)
        n_c_val = max(1, len(c_idx) // 5)
        val_indices.extend(lt_indices[c_idx[:n_c_val]])
        train_indices.extend(lt_indices[c_idx[n_c_val:]])
    
    # Transform 설정
    train_tf = transforms.Compose([
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.4914, 0.4822, 0.4465],
                             std=[0.2023, 0.1994, 0.2010]),
    ])
    test_tf = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.4914, 0.4822, 0.4465],
                             std=[0.2023, 0.1994, 0.2010]),
    ])
    
    # Datasets with transform
    train_ds = Subset(full_dataset, train_indices)
    train_ds.dataset.transform = train_tf
    
    val_ds = Subset(full_dataset, val_indices)
    val_ds.dataset.transform = test_tf
    
    test_ds = datasets.CIFAR10(root='/tmp/cifar', train=False, download=True, transform=test_tf)
    
    # DataLoaders
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=num_workers)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=num_workers)
    test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False, num_workers=num_workers)
    
    return train_loader, val_loader, test_loader, class_counts


print('✓ CIFAR-LT 함수 정의 완료')

✓ CIFAR-LT 함수 정의 완료


In [3]:
# === Cell 2: 클래스 분포 시각화 ===

def visualize_class_distribution(class_counts, ir, dataset_name='CIFAR-10'):
    """
    불균형 분포 시각화 및 그룹 경계 출력.
    """
    counts_arr = np.array(class_counts)
    
    print(f'\n{"="*60}')
    print(f'{dataset_name} LT (IR={ir}) — 클래스 분포')
    print(f'{"="*60}')
    print(f'Min: {counts_arr.min():5d} | Max: {counts_arr.max():5d} | Ratio: {counts_arr.max()/counts_arr.min():.1f}:1')
    print(f'Total samples: {counts_arr.sum():,}')
    
    # Many/Medium/Few 그룹 계산
    many_mask = counts_arr >= 100
    medium_mask = (counts_arr >= 20) & (counts_arr < 100)
    few_mask = counts_arr < 20
    
    print(f'\nGroup distribution:')
    print(f'  Many-shot (n≥100):   {many_mask.sum():2d} classes')
    print(f'  Medium-shot (20≤n):  {medium_mask.sum():2d} classes')
    print(f'  Few-shot (n<20):     {few_mask.sum():2d} classes')
    
    # Bar chart
    fig, ax = plt.subplots(figsize=(12, 4))
    colors = ['green' if m else ('orange' if med else 'red') 
              for m, med in zip(many_mask, medium_mask)]
    ax.bar(range(len(class_counts)), class_counts, color=colors, alpha=0.7)
    ax.set_xlabel('Class')
    ax.set_ylabel('# Samples (log scale)', fontsize=11)
    ax.set_yscale('log')
    ax.set_title(f'{dataset_name} LT Distribution (IR={ir})', fontsize=13, fontweight='bold')
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    
    ir_dir = f'{RESULTS_BASE}/IR{ir}'
    os.makedirs(ir_dir, exist_ok=True)
    plt.savefig(f'{ir_dir}/class_distribution.png', dpi=100, bbox_inches='tight')
    plt.close()
    print(f'\n✓ 분포 시각화 저장: {ir_dir}/class_distribution.png')


# Test with first IR
for ir in IR_LIST:
    os.makedirs(f'{RESULTS_BASE}/IR{ir}', exist_ok=True)
    _, _, _, class_counts = load_cifar_lt_loaders(ir, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS)
    visualize_class_distribution(class_counts, ir)

100%|██████████| 170M/170M [00:09<00:00, 17.6MB/s] 



CIFAR-10 LT (IR=10) — 클래스 분포
Min:   500 | Max:  5000 | Ratio: 10.0:1
Total samples: 20,431

Group distribution:
  Many-shot (n≥100):   10 classes
  Medium-shot (20≤n):   0 classes
  Few-shot (n<20):      0 classes

✓ 분포 시각화 저장: /content/drive/MyDrive/imbalanced-data-LWCE/image_classification/results/CIFAR10_LT/IR10/class_distribution.png

CIFAR-10 LT (IR=50) — 클래스 분포
Min:    99 | Max:  5000 | Ratio: 50.5:1
Total samples: 13,995

Group distribution:
  Many-shot (n≥100):    9 classes
  Medium-shot (20≤n):   1 classes
  Few-shot (n<20):      0 classes

✓ 분포 시각화 저장: /content/drive/MyDrive/imbalanced-data-LWCE/image_classification/results/CIFAR10_LT/IR50/class_distribution.png

CIFAR-10 LT (IR=100) — 클래스 분포
Min:    50 | Max:  5000 | Ratio: 100.0:1
Total samples: 12,406

Group distribution:
  Many-shot (n≥100):    8 classes
  Medium-shot (20≤n):   2 classes
  Few-shot (n<20):      0 classes

✓ 분포 시각화 저장: /content/drive/MyDrive/imbalanced-data-LWCE/image_classification/results/CIFAR10_LT/IR1

In [ ]:
# === Cell 3: 모델 및 평가 함수 정의 ===

def compute_val_metrics(model, loader, num_classes, class_counts_train=None):
    """
    Balanced Accuracy, F1-Macro, Many/Medium/Few-shot accuracy 계산.
    그룹: training class count 기준 상위/중위/하위 1/3 (tertile split).
    """
    model.eval()
    cm = torch.zeros(num_classes, num_classes, dtype=torch.long)

    with torch.no_grad():
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)
            preds = model(imgs).argmax(dim=1)
            for t, p in zip(labels.view(-1), preds.view(-1)):
                cm[t.long(), p.long()] += 1

    # Per-class accuracy
    per_class_acc = (cm.diagonal().float() / cm.sum(1).clamp(min=1).float()).cpu().numpy()
    balanced_acc = float(per_class_acc.mean())
    top1_acc = float(cm.diagonal().sum() / cm.sum())

    # F1-Macro (reconstruct y_true, y_pred from cm)
    y_true, y_pred = [], []
    for i in range(num_classes):
        for j in range(num_classes):
            y_true.extend([i] * cm[i, j].item())
            y_pred.extend([j] * cm[i, j].item())
    f1_macro = f1_score(y_true, y_pred, average='macro', zero_division=0)

    result = {
        'Top1_Acc': top1_acc,
        'Balanced_Acc': balanced_acc,
        'F1_Macro': f1_macro,
        'Per_Class_Acc': per_class_acc.tolist(),
    }

    if class_counts_train is not None:
        counts = np.array(class_counts_train)
        sorted_idx = np.argsort(counts)[::-1]   # 내림차순 (many→few)
        n = len(sorted_idx)
        many_idx   = sorted_idx[:n // 3]
        medium_idx = sorted_idx[n // 3 : 2 * n // 3]
        few_idx    = sorted_idx[2 * n // 3:]

        result['Many_Acc']   = float(per_class_acc[many_idx].mean())
        result['Medium_Acc'] = float(per_class_acc[medium_idx].mean())
        result['Few_Acc']    = float(per_class_acc[few_idx].mean())

    return result


def compute_val_acc(model, loader):
    """Fast scalar for Optuna proxy and train_model checkpoint (F1-Macro)."""
    model.eval()
    all_preds, all_labels = [], []

    with torch.no_grad():
        for imgs, labels in loader:
            imgs = imgs.to(device)
            preds = model(imgs).argmax(dim=1).cpu()
            all_preds.extend(preds.tolist())
            all_labels.extend(labels.tolist())

    return f1_score(all_labels, all_preds, average='macro', zero_division=0)


print('✓ 모델 및 평가 함수 정의 완료 (Optuna/checkpoint 기준: F1-Macro, 그룹: tertile split)')

✓ 모델 및 평가 함수 정의 완료 (Optuna/checkpoint 기준: F1-Macro, 그룹: tertile split)


In [ ]:
# === Cell 4: train_model 함수 정의 ===

def train_model(loss_name: str,
                class_counts: list,
                train_loader,
                val_loader,
                num_classes: int,
                alpha: float = 1.0,
                gamma: float = 2.0,
                epochs: int = 200,
                lr: float = 0.1,
                tag: str = '',
                seed: int = 42):

    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)

    model = build_resnet32(num_classes).to(device)

    optimizer = optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=2e-4)
    scheduler = optim.lr_scheduler.MultiStepLR(optimizer, milestones=[160, 180], gamma=0.01)

    criterion = get_clf_loss(loss_name, class_counts, alpha=alpha, gamma=gamma)

    best_val_acc = 0.0
    best_model_state = None
    history = {'epoch': [], 'train_loss': [], 'val_acc': [], 'val_balanced_acc': []}

    pbar = tqdm(range(epochs), desc=f'{loss_name} (α={alpha:.2f}, γ={gamma:.2f})', leave=False)

    for epoch in pbar:
        model.train()
        train_loss = 0.0
        for imgs, labels in train_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            optimizer.zero_grad()
            logits = model(imgs)
            loss = criterion(logits, labels)
            loss.backward()
            optimizer.step()
            train_loss += loss.item() * labels.size(0)

        train_loss /= len(train_loader.dataset)

        val_bal_acc = compute_val_acc(model, val_loader)

        history['epoch'].append(epoch)
        history['train_loss'].append(train_loss)
        history['val_balanced_acc'].append(val_bal_acc)

        if val_bal_acc > best_val_acc:
            best_val_acc = val_bal_acc
            best_model_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

        scheduler.step()
        pbar.update()

    if best_model_state:
        model.load_state_dict(best_model_state)
    model.eval()

    return model, history, best_val_acc


print('✓ train_model 함수 정의 완료')

✓ train_model 함수 정의 완료


In [6]:
# === Cell 5: Optuna alpha/gamma 탐색 ===

os.environ['TQDM_DISABLE'] = '1'

PROXY_EPOCHS = 20
PROXY_SUBSET_RATIO = 0.20
N_TRIALS_PWCE  = 20
N_TRIALS_PLWCE = 20
N_TRIALS_FOCAL = 20
ALPHA_LOW, ALPHA_HIGH = 1.0, 15.0
PWCE_LOW,  PWCE_HIGH  = 0.5, 5.0
GAMMA_LOW, GAMMA_HIGH = 0.5, 5.0

# ── 체크포인트 로드 ──────────────────────────────────
if os.path.exists(CKPT_OPTUNA):
    with open(CKPT_OPTUNA) as f:
        optuna_best = {int(k): v for k, v in json.load(f).items()}
    print(f'Optuna 체크포인트 로드: IR={sorted(optuna_best.keys())} 완료됨')
    for ir, best in sorted(optuna_best.items()):
        print(f'  IR={ir}: pwce α={best["pwce"]["alpha"]:.3f}, '
              f'plwce α={best["plwce"]["alpha"]:.3f}, '
              f'focal γ={best["focal"]["gamma"]:.3f}')
else:
    optuna_best = {}
    print('Optuna 체크포인트 없음 — 새로 시작')

print(f'\nOptuna 탐색 시작 (proxy: {PROXY_EPOCHS} epochs, subset={PROXY_SUBSET_RATIO})')
print('=' * 60)

for ir in IR_LIST:
    if ir in optuna_best:
        print(f'[IR={ir}] 스킵 (완료됨)')
        continue

    print(f'\n[IR={ir}] Optuna 탐색 중...')

    train_loader, val_loader, _, class_counts = load_cifar_lt_loaders(
        ir, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS)

    n_subset = max(1, int(len(train_loader.dataset) * PROXY_SUBSET_RATIO))
    subset_indices = np.random.choice(len(train_loader.dataset), size=n_subset, replace=False)
    proxy_train_ds = Subset(train_loader.dataset, subset_indices)
    proxy_train_loader = DataLoader(proxy_train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)

    def objective_pwce(trial):
        alpha = trial.suggest_float('alpha', PWCE_LOW, PWCE_HIGH)
        model, _, _ = train_model('pwce', class_counts, proxy_train_loader, val_loader,
                                   NUM_CLASSES, alpha=alpha, epochs=PROXY_EPOCHS, tag='optuna_pwce')
        return compute_val_acc(model, val_loader)

    pwce_alphas = [0.5, 1.0] + np.linspace(PWCE_LOW, PWCE_HIGH, N_TRIALS_PWCE - 2).tolist()
    study_pwce = optuna.create_study(direction='maximize',
                                     sampler=optuna.samplers.GridSampler({'alpha': pwce_alphas}),
                                     study_name=f'cifar10_ir{ir}_pwce')
    study_pwce.optimize(objective_pwce, n_trials=N_TRIALS_PWCE, show_progress_bar=False)

    def objective_plwce(trial):
        alpha = trial.suggest_float('alpha', ALPHA_LOW, ALPHA_HIGH)
        model, _, _ = train_model('plwce', class_counts, proxy_train_loader, val_loader,
                                   NUM_CLASSES, alpha=alpha, epochs=PROXY_EPOCHS, tag='optuna_plwce')
        return compute_val_acc(model, val_loader)

    plwce_alphas = [0.5, 1.0] + np.linspace(ALPHA_LOW, ALPHA_HIGH, N_TRIALS_PLWCE - 2).tolist()
    study_plwce = optuna.create_study(direction='maximize',
                                      sampler=optuna.samplers.GridSampler({'alpha': plwce_alphas}),
                                      study_name=f'cifar10_ir{ir}_plwce')
    study_plwce.optimize(objective_plwce, n_trials=N_TRIALS_PLWCE, show_progress_bar=False)

    def objective_focal(trial):
        gamma = trial.suggest_float('gamma', GAMMA_LOW, GAMMA_HIGH)
        model, _, _ = train_model('focal', class_counts, proxy_train_loader, val_loader,
                                   NUM_CLASSES, gamma=gamma, epochs=PROXY_EPOCHS, tag='optuna_focal')
        return compute_val_acc(model, val_loader)

    focal_gammas = np.linspace(GAMMA_LOW, GAMMA_HIGH, N_TRIALS_FOCAL).tolist()
    study_focal = optuna.create_study(direction='maximize',
                                      sampler=optuna.samplers.GridSampler({'gamma': focal_gammas}),
                                      study_name=f'cifar10_ir{ir}_focal')
    study_focal.optimize(objective_focal, n_trials=N_TRIALS_FOCAL, show_progress_bar=False)

    optuna_best[ir] = {
        'pwce':  {'alpha': study_pwce.best_params['alpha']},
        'plwce': {'alpha': study_plwce.best_params['alpha']},
        'focal': {'gamma': study_focal.best_params['gamma']},
    }

    print(f'  PWCE  best α={optuna_best[ir]["pwce"]["alpha"]:.3f}')
    print(f'  PLWCE best α={optuna_best[ir]["plwce"]["alpha"]:.3f}')
    print(f'  Focal best γ={optuna_best[ir]["focal"]["gamma"]:.3f}')

    with open(CKPT_OPTUNA, 'w') as f:
        json.dump({str(k): v for k, v in optuna_best.items()}, f, indent=2)
    print(f'  체크포인트 저장 → {CKPT_OPTUNA}')

os.environ['TQDM_DISABLE'] = '0'
print(f'\n✓ Optuna 탐색 완료 — pwce, plwce, focal 최적화됨')
print(f'  (ce, lwce, cb는 기본값 사용)')

Optuna 체크포인트 로드: IR=[10, 50, 100] 완료됨
  IR=10: pwce α=0.765, plwce α=5.941, focal γ=2.158
  IR=50: pwce α=0.500, plwce α=2.647, focal γ=1.211
  IR=100: pwce α=0.500, plwce α=2.647, focal γ=1.211

Optuna 탐색 시작 (proxy: 20 epochs, subset=0.2)
[IR=10] 스킵 (완료됨)
[IR=50] 스킵 (완료됨)
[IR=100] 스킵 (완료됨)

✓ Optuna 탐색 완료 — pwce, plwce, focal 최적화됨
  (ce, lwce, cb는 기본값 사용)


In [7]:
# === Cell 6: 전체 Loss × IR × Seed 비교 실험 ===

LOSS_CONFIGS = ['ce', 'pwce', 'lwce', 'plwce', 'cb', 'focal']

# ── 체크포인트 로드 ──────────────────────────────────
if os.path.exists(CKPT_RESULTS):
    with open(CKPT_RESULTS) as f:
        ckpt_data = json.load(f)
    print(f'결과 체크포인트 로드: {len(ckpt_data)}개 완료된 실행')
    for k in sorted(ckpt_data.keys()):
        m = ckpt_data[k]['metrics']
        print(f'  [{k}] Top1={m["Top1_Acc"]:.4f} F1={m["F1_Macro"]:.4f}')
else:
    ckpt_data = {}
    print('체크포인트 없음 — 새로 시작')

# ── 완료된 결과 복원 ────────────────────────────────
all_results   = {ir: {l: {} for l in LOSS_CONFIGS} for ir in IR_LIST}
all_histories = {ir: {l: {} for l in LOSS_CONFIGS} for ir in IR_LIST}

for run_key, run_data in ckpt_data.items():
    # run_key 형식: IR{ir}_{loss_name}_s{seed}
    try:
        ir_str, loss_str, seed_str = run_key.split('_', 2)
        ir   = int(ir_str[2:])
        seed = int(seed_str[1:])
        if ir in all_results and loss_str in all_results[ir]:
            all_results[ir][loss_str][seed]   = run_data['metrics']
            all_histories[ir][loss_str][seed] = run_data['history']
    except Exception:
        pass

total_runs = len(IR_LIST) * len(LOSS_CONFIGS) * len(SEEDS)
print(f'\nFull experiment: {len(IR_LIST)} IRs × {len(LOSS_CONFIGS)} losses × {len(SEEDS)} seeds = {total_runs} runs')
print(f'완료: {len(ckpt_data)}/{total_runs}')
print('=' * 60)

# ── 실험 실행 ────────────────────────────────────────
for ir in IR_LIST:
    train_loader, val_loader, test_loader, class_counts = load_cifar_lt_loaders(
        ir, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS)

    for loss_name in LOSS_CONFIGS:
        alpha = optuna_best[ir].get(loss_name, {}).get('alpha', 1.0)
        gamma = optuna_best[ir].get('focal', {}).get('gamma', 2.0) if loss_name == 'focal' else 2.0

        for seed in SEEDS:
            run_key = f'IR{ir}_{loss_name}_s{seed}'
            if run_key in ckpt_data:
                continue

            print(f'\n[IR={ir}] {loss_name} seed={seed}...')

            model, history, _ = train_model(
                loss_name, class_counts, train_loader, val_loader,
                NUM_CLASSES, alpha=alpha, gamma=gamma,
                epochs=FINAL_EPOCHS, tag=f'ir{ir}_{loss_name}_s{seed}', seed=seed)

            metrics = compute_val_metrics(model, test_loader, NUM_CLASSES, class_counts_train=class_counts)
            metrics['alpha'] = alpha
            metrics['gamma'] = gamma

            all_results[ir][loss_name][seed]   = metrics
            all_histories[ir][loss_name][seed] = history

            ckpt_data[run_key] = {'metrics': metrics, 'history': history}
            with open(CKPT_RESULTS, 'w') as f:
                json.dump(ckpt_data, f)

            print(f'  Top1={metrics["Top1_Acc"]:.4f} | F1={metrics["F1_Macro"]:.4f} | '
                  f'Few={metrics.get("Few_Acc", 0):.4f} → 저장됨')

print(f'\n✓ 전체 훈련 완료')

결과 체크포인트 로드: 56개 완료된 실행
  [IR100_cb_s42] Top1=0.4562 F1=0.4121
  [IR100_cb_s43] Top1=0.4421 F1=0.4005
  [IR100_cb_s44] Top1=0.4697 F1=0.4327
  [IR100_ce_s42] Top1=0.4683 F1=0.4295
  [IR100_ce_s43] Top1=0.4667 F1=0.4174
  [IR100_ce_s44] Top1=0.4838 F1=0.4434
  [IR100_focal_s42] Top1=0.4737 F1=0.4381
  [IR100_focal_s43] Top1=0.4926 F1=0.4469
  [IR100_focal_s44] Top1=0.4978 F1=0.4612
  [IR100_lwce_s42] Top1=0.4882 F1=0.4396
  [IR100_lwce_s43] Top1=0.4833 F1=0.4399
  [IR100_lwce_s44] Top1=0.4979 F1=0.4566
  [IR100_plwce_s42] Top1=0.4918 F1=0.4522
  [IR100_plwce_s43] Top1=0.5208 F1=0.4874
  [IR100_plwce_s44] Top1=0.4818 F1=0.4377
  [IR100_pwce_s42] Top1=0.4897 F1=0.4448
  [IR100_pwce_s43] Top1=0.4734 F1=0.4287
  [IR100_pwce_s44] Top1=0.4943 F1=0.4569
  [IR10_cb_s42] Top1=0.7356 F1=0.7374
  [IR10_cb_s43] Top1=0.7288 F1=0.7301
  [IR10_cb_s44] Top1=0.7295 F1=0.7313
  [IR10_ce_s42] Top1=0.7102 F1=0.7112
  [IR10_ce_s43] Top1=0.7077 F1=0.7083
  [IR10_ce_s44] Top1=0.7194 F1=0.7207
  [IR10_ce_s45] 

pwce (α=0.76, γ=2.00):   0%|          | 0/200 [00:00<?, ?it/s]

  Top1=0.7086 | F1=0.7112 | Few=0.6608 → 저장됨

[IR=10] pwce seed=46...


pwce (α=0.76, γ=2.00):   0%|          | 0/200 [00:00<?, ?it/s]

  Top1=0.7371 | F1=0.7393 | Few=0.7042 → 저장됨

[IR=10] lwce seed=45...


...

  Top1=0.4377 | F1=0.3958 | Few=0.1790 → 저장됨

[IR=100] focal seed=45...


focal (α=1.00, γ=1.21):   0%|          | 0/200 [00:00<?, ?it/s]

  Top1=0.4952 | F1=0.4584 | Few=0.2298 → 저장됨

[IR=100] focal seed=46...


focal (α=1.00, γ=1.21):   0%|          | 0/200 [00:00<?, ?it/s]

  Top1=0.5090 | F1=0.4779 | Few=0.2720 → 저장됨

✓ 전체 훈련 완료


In [8]:
# === Cell 7: 결과 집계, 시각화 및 저장 ===

def compute_class_counts_pure(dataset_name, ir):
    """데이터 다운로드 없이 class_counts 계산."""
    K = 10 if dataset_name == 'cifar10' else 100
    n_max = 5000 if dataset_name == 'cifar10' else 500
    rho = ir ** (-1 / (K - 1))
    return [int(n_max * (rho ** i)) for i in range(K)]

def get_group_indices(class_counts):
    """Training count 기준 상위/중위/하위 1/3 인덱스 반환."""
    counts = np.array(class_counts)
    sorted_idx = np.argsort(counts)[::-1]
    n = len(sorted_idx)
    return sorted_idx[:n // 3], sorted_idx[n // 3 : 2 * n // 3], sorted_idx[2 * n // 3:]

print(f'\n최종 결과 요약 (mean ± std, 그룹=상위/중위/하위 1/3 tertile split)')
print('=' * 105)

agg_results = {}
colors_loss = plt.cm.tab10(np.linspace(0, 1, len(LOSS_CONFIGS)))

for ir in IR_LIST:
    agg_results[ir] = {}
    cc_ir = compute_class_counts_pure(DATASET, ir)
    many_idx, medium_idx, few_idx = get_group_indices(cc_ir)

    print(f'\nIR={ir}:')
    print(f'{"Loss":15s} | {"Top1":^16} | {"Balanced":^16} | {"F1-Macro":^16} | {"Many":^8} | {"Medium":^8} | {"Few":^8} | n')
    print('-' * 115)

    for loss_name in LOSS_CONFIGS:
        seed_metrics = [all_results[ir][loss_name][s]
                        for s in SEEDS if s in all_results[ir][loss_name]]
        if not seed_metrics:
            print(f'{loss_name:15s} | (미완료)')
            continue

        # Per_Class_Acc로 그룹별 acc 재계산 (체크포인트 구버전 호환)
        for m in seed_metrics:
            pca = np.array(m.get('Per_Class_Acc', []))
            if len(pca) > 0:
                m['Many_Acc']   = float(pca[many_idx].mean())
                m['Medium_Acc'] = float(pca[medium_idx].mean())
                m['Few_Acc']    = float(pca[few_idx].mean())

        agg = {}
        for key in ['Top1_Acc', 'Balanced_Acc', 'F1_Macro', 'Many_Acc', 'Medium_Acc', 'Few_Acc']:
            vals = [m.get(key, 0.0) for m in seed_metrics]
            agg[f'{key}_mean'] = float(np.mean(vals))
            agg[f'{key}_std']  = float(np.std(vals))
        agg_results[ir][loss_name] = agg

        n = len(seed_metrics)
        print(f'{loss_name:15s} | '
              f'{agg["Top1_Acc_mean"]:.4f}±{agg["Top1_Acc_std"]:.4f} | '
              f'{agg["Balanced_Acc_mean"]:.4f}±{agg["Balanced_Acc_std"]:.4f} | '
              f'{agg["F1_Macro_mean"]:.4f}±{agg["F1_Macro_std"]:.4f} | '
              f'{agg["Many_Acc_mean"]:.4f}   | '
              f'{agg["Medium_Acc_mean"]:.4f}   | '
              f'{agg["Few_Acc_mean"]:.4f}   | {n}')

    # ── JSON 저장 ──────────────────────────────────────
    os.makedirs(f'{RESULTS_BASE}/IR{ir}', exist_ok=True)
    with open(f'{RESULTS_BASE}/IR{ir}/results_agg.json', 'w') as f:
        json.dump(agg_results[ir], f, indent=2)
    per_seed_data = {l: {str(s): all_results[ir][l].get(s, {}) for s in SEEDS} for l in LOSS_CONFIGS}
    with open(f'{RESULTS_BASE}/IR{ir}/results_per_seed.json', 'w') as f:
        json.dump(per_seed_data, f, indent=2)

    # ── Excel 저장 ────────────────────────────────────
    summary_rows, per_seed_rows = [], []
    for loss_name in LOSS_CONFIGS:
        if loss_name not in agg_results[ir]:
            continue
        agg = agg_results[ir][loss_name]
        summary_rows.append({
            'Loss':          loss_name,
            'Top1_Mean':     f"{agg['Top1_Acc_mean']:.4f}",
            'Top1_Std':      f"{agg['Top1_Acc_std']:.4f}",
            'Balanced_Mean': f"{agg['Balanced_Acc_mean']:.4f}",
            'Balanced_Std':  f"{agg['Balanced_Acc_std']:.4f}",
            'F1_Macro_Mean': f"{agg['F1_Macro_mean']:.4f}",
            'F1_Macro_Std':  f"{agg['F1_Macro_std']:.4f}",
            'Many_Mean':     f"{agg['Many_Acc_mean']:.4f}",
            'Medium_Mean':   f"{agg['Medium_Acc_mean']:.4f}",
            'Few_Mean':      f"{agg['Few_Acc_mean']:.4f}",
        })
        for seed in SEEDS:
            m = all_results[ir][loss_name].get(seed, {})
            if m:
                per_seed_rows.append({
                    'Loss': loss_name, 'Seed': seed,
                    'Top1_Acc':     f"{m['Top1_Acc']:.4f}",
                    'Balanced_Acc': f"{m['Balanced_Acc']:.4f}",
                    'F1_Macro':     f"{m['F1_Macro']:.4f}",
                    'Many_Acc':     f"{m.get('Many_Acc', 0):.4f}",
                    'Medium_Acc':   f"{m.get('Medium_Acc', 0):.4f}",
                    'Few_Acc':      f"{m.get('Few_Acc', 0):.4f}",
                })

    with pd.ExcelWriter(f'{RESULTS_BASE}/IR{ir}/results.xlsx', engine='openpyxl') as writer:
        pd.DataFrame(summary_rows).to_excel(writer, sheet_name='Summary_Agg', index=False)
        pd.DataFrame(per_seed_rows).to_excel(writer, sheet_name='Per_Seed', index=False)

# ── 학습 곡선 (mean ± std shading) ─────────────────────
fig, axes = plt.subplots(1, len(IR_LIST), figsize=(15, 4))
if len(IR_LIST) == 1:
    axes = [axes]

for ax_idx, ir in enumerate(IR_LIST):
    ax = axes[ax_idx]
    for l_idx, loss_name in enumerate(LOSS_CONFIGS):
        seed_hists = [all_histories[ir][loss_name][s]
                      for s in SEEDS if s in all_histories[ir][loss_name]]
        if not seed_hists:
            continue
        arr  = np.array([h['val_balanced_acc'] for h in seed_hists])
        mean = arr.mean(0)
        std  = arr.std(0)
        ep   = np.arange(len(mean))
        ax.plot(ep, mean, label=loss_name, color=colors_loss[l_idx], alpha=0.85)
        ax.fill_between(ep, mean - std, mean + std, color=colors_loss[l_idx], alpha=0.15)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Val F1-Macro')
    ax.set_title(f'IR={ir}')
    ax.legend(fontsize=7)
    ax.grid(alpha=0.3)

plt.suptitle(f'{DATASET.upper()}-LT Training Curves (mean±std, n≤{len(SEEDS)} seeds)', fontsize=12)
plt.tight_layout()
plt.savefig(f'{RESULTS_BASE}/training_curves_all_irs.png', dpi=150, bbox_inches='tight')
plt.close()
print('✓ 학습 곡선 저장')

# ── F1-Macro 비교 막대 그래프 ────────────────────────
fig, axes = plt.subplots(1, len(IR_LIST), figsize=(15, 4), sharey=False)
if len(IR_LIST) == 1:
    axes = [axes]

for ax_idx, ir in enumerate(IR_LIST):
    ax = axes[ax_idx]
    losses = [l for l in LOSS_CONFIGS if l in agg_results[ir]]
    means  = [agg_results[ir][l]['F1_Macro_mean'] for l in losses]
    stds   = [agg_results[ir][l]['F1_Macro_std']  for l in losses]
    x = np.arange(len(losses))
    ax.bar(x, means, yerr=stds, capsize=5, alpha=0.78,
           color=[colors_loss[LOSS_CONFIGS.index(l)] for l in losses])
    ce_mean = agg_results[ir].get('ce', {}).get('F1_Macro_mean')
    if ce_mean is not None:
        ax.axhline(ce_mean, color='gray', linestyle='--', linewidth=1,
                   alpha=0.7, label='CE baseline')
        ax.legend(fontsize=7)
    ax.set_xticks(x)
    ax.set_xticklabels(losses, rotation=30, ha='right')
    ax.set_ylabel('F1-Macro')
    ax.set_title(f'IR={ir}')
    ax.grid(axis='y', alpha=0.3)

plt.suptitle(f'{DATASET.upper()}-LT F1-Macro (mean±std, n≤{len(SEEDS)} seeds)', fontsize=12)
plt.tight_layout()
plt.savefig(f'{RESULTS_BASE}/f1_macro_comparison.png', dpi=150, bbox_inches='tight')
plt.close()
print('✓ F1-Macro 비교 그래프 저장')

# ── Many/Medium/Few 그룹별 정확도 비교 ──────────────────
fig, axes = plt.subplots(1, len(IR_LIST), figsize=(15, 4))
if len(IR_LIST) == 1:
    axes = [axes]

width = 0.25
group_colors = {'Many': '#2ecc71', 'Medium': '#f39c12', 'Few': '#e74c3c'}

for ax_idx, ir in enumerate(IR_LIST):
    ax = axes[ax_idx]
    losses = [l for l in LOSS_CONFIGS if l in agg_results[ir]]
    x = np.arange(len(losses))
    for g_idx, (group, key) in enumerate([('Many',   'Many_Acc_mean'),
                                           ('Medium', 'Medium_Acc_mean'),
                                           ('Few',    'Few_Acc_mean')]):
        vals = [agg_results[ir][l].get(key, 0.0) for l in losses]
        ax.bar(x + (g_idx - 1) * width, vals, width, label=group,
               color=group_colors[group], alpha=0.78)
    ax.set_xticks(x)
    ax.set_xticklabels(losses, rotation=30, ha='right')
    ax.set_ylabel('Accuracy')
    ax.set_title(f'IR={ir}')
    ax.legend(fontsize=8)
    ax.grid(axis='y', alpha=0.3)

plt.suptitle(f'{DATASET.upper()}-LT Many/Medium/Few Accuracy (tertile split)', fontsize=12)
plt.tight_layout()
plt.savefig(f'{RESULTS_BASE}/group_accuracy_comparison.png', dpi=150, bbox_inches='tight')
plt.close()
print('✓ 그룹별 정확도 그래프 저장')

print(f'\n✓ 모든 결과 저장 완료')
print(f'  JSON (집계):   {RESULTS_BASE}/IR*/results_agg.json')
print(f'  JSON (seed별): {RESULTS_BASE}/IR*/results_per_seed.json')
print(f'  Excel:         {RESULTS_BASE}/IR*/results.xlsx  (Summary_Agg / Per_Seed 시트)')
print(f'  Checkpoint:    {CKPT_RESULTS}')
print(f'  PNG: training_curves / f1_macro_comparison / group_accuracy_comparison')


최종 결과 요약 (mean ± std, 그룹=상위/중위/하위 1/3 tertile split)

IR=10:
Loss            |       Top1       |     Balanced     |     F1-Macro     |   Many   |  Medium  |   Few    | n
-------------------------------------------------------------------------------------------------------------------
ce              | 0.7092±0.0077 | 0.7092±0.0077 | 0.7103±0.0074 | 0.8601   | 0.6399   | 0.6479   | 5
pwce            | 0.7212±0.0093 | 0.7212±0.0093 | 0.7235±0.0092 | 0.8536   | 0.6384   | 0.6841   | 5
lwce            | 0.7166±0.0101 | 0.7166±0.0101 | 0.7182±0.0100 | 0.8556   | 0.6451   | 0.6659   | 5
plwce           | 0.7250±0.0152 | 0.7250±0.0152 | 0.7268±0.0154 | 0.8520   | 0.6459   | 0.6891   | 5
cb              | 0.7278±0.0103 | 0.7278±0.0103 | 0.7296±0.0104 | 0.8499   | 0.6470   | 0.6969   | 5
focal           | 0.7121±0.0158 | 0.7121±0.0158 | 0.7143±0.0160 | 0.8615   | 0.6468   | 0.6490   | 5

IR=50:
Loss            |       Top1       |     Balanced     |     F1-Macro     |   Many   |  Medium  |  